# File 1: Establishing Cohort with Geopermission & demographics


### This file establishes permission to link for people with geolinkage permission, and their harmonised sociodemograhics from linked health data and cohort data 


In [ ]:
import pandas as pd


In [ ]:
# read df
df = pd.read_stata("S:\LLC_0002\lamj\Datasets\Participant Base\Permission_Status_Base.dta")


In [ ]:
# keep if llc status == 1
df = df[df['ukllc_status'] == 1]

In [ ]:
# establish linkage status
df['participant_link'] = 0 
df.loc[df['NHSD_linked'] != "", 'participant_link'] = 1

In [ ]:
# define consent type
df['consent_type'] = 0
df.loc[(df['nhs_e_linkage_permission'] == "")|(df['nhs_e_linkage_permission'] == "0") | (df['nhs_e_linkage_permission'] == "NULL"), 'consent_type'] = 0
df.loc[(df['nhs_e_linkage_permission'] == "1") & (df['national_opt_out'] == 1), 'consent_type'] = 1
df.loc[(df['nhs_e_linkage_permission'] == "1") & (df['national_opt_out'] == 0), 'consent_type'] = 2
df['any_consent'] = 0
df.loc[(df['consent_type'] == 1) | (df['consent_type'] == 2), 'any_consent'] = 1


In [ ]:
# permission to link
df['permission_link'] = 0
df.loc[df['nhs_e_linkage_permission'] == "1", 'permission_link'] = 1
df['permission_link'].value_counts()

In [ ]:
# keep only with permission to link
df = df[df['permission_link'] == 1]

# keep relvant columns
columns = ['LLC_0002_stud_id','cohort','geocoding_permission','participant_link','consent_type','any_consent']
df = df[columns]

In [ ]:
df['geocoding_permission'].value_counts()

In [ ]:
df['participant_link'].value_counts()

In [ ]:
demo_df = df[(df['geocoding_permission'] != "0")]
demo_df = demo_df[(demo_df['geocoding_permission'] != "NULL")]
demo_df = demo_df.rename(columns = {"LLC_0002_stud_id": "llc_0002_stud_id"})

In [ ]:
# linking with demographic variables (dated)
# best measure sex, ethnic 7, age

# re-run this with access to up-to-date self-reported data.

bestMeasure_sex = pd.read_stata(r'S:\LLC_0002\lamj\Datasets\Multiple-Source_Harmonisation\NHS_&_Self-Report_BestMeasure_Sex.dta')
bestMeasure_ethnic7 = pd.read_stata(r'S:\LLC_0002\lamj\Datasets\Multiple-Source_Harmonisation\NHS_&_Self-Report_BestMeasure_ethnic7.dta')
bestMeasure_age = pd.read_stata(r'S:\LLC_0002\lamj\Datasets\Multiple-Source_Harmonisation\NHS_&_Self-Report_BestMeasure_age.dta')

merged_sex = pd.merge(df,bestMeasure_sex[['LLC_0002_stud_id', "value", 'label']], on = "LLC_0002_stud_id", how = 'left')
merged_sex.rename(columns = {'value': 'sex'}, inplace = True)
merged_sex['sex'].replace({8:"Male",9:"Female",99:pd.NA}, inplace = True)
merged_sex.drop(columns=['label'], inplace = True)

merged_sex_ethnic = pd.merge(merged_sex,bestMeasure_ethnic7[['LLC_0002_stud_id', "value", 'label']], on = "LLC_0002_stud_id", how = 'left')
merged_sex_ethnic.rename(columns = {'value': 'ethnicity'}, inplace = True)
merged_sex_ethnic['ethnicity'].replace({0:"White",1:"Black",2:"South-east Asian",3:"Other Asian", 4:"Mixed",5:"Other",99:pd.NA}, inplace = True)
merged_sex_ethnic.drop(columns=['label'], inplace = True)

merged_sex_ethnic_age = pd.merge(merged_sex_ethnic,bestMeasure_age[['LLC_0002_stud_id', "value", 'label']], on = "LLC_0002_stud_id",how = 'left')
merged_sex_ethnic_age.rename(columns = {'value': 'age'}, inplace = True)
merged_sex_ethnic_age['age'].replace({13:"<=18 years",14:"19-30 years",15:"31-59 years",16:"60-74 years", 17:"75+ years"}, inplace = True)
merged_sex_ethnic_age.drop(columns=['label'], inplace = True)

merged_df = merged_sex_ethnic_age
merged_df = merged_df.rename(columns = {"LLC_0002_stud_id": "llc_0002_stud_id"})

In [ ]:
cohort = pd.merge(demo_df, merged_df, on = "llc_0002_stud_id", how = "left", suffixes = ('','_drop'))
col_to_drop = [c for c in cohort.columns if c.endswith('_drop')]
cohort.drop(columns=col_to_drop, inplace=True)


In [ ]:
cohort.to_csv(r'S:\LLC_0002\lamj\notebooks_lsoa\LSOA\demographics_geo.csv')

In [ ]:
cohort['geocoding_permission'].value_counts()

In [ ]:
cohort[cohort['geocoding_permission'] != "Null"]['cohort'].value_counts()